# World Cup 2026 Simulation
--------------------------------
- Simulate the world cup group stage table
- Add logic for draws
- Add human intuitive mentrics

#Import Necessarry Libraries
------------------

In [0]:
from pyspark.sql.functions import *

# Parameters
-----------------

In [0]:
catalog = 'wc_2026_predions'
gold_schema = 'gold'
gold_table_name = 'internation_results_data'
pred_table_name = 'future_game_predictions'

#Read in Predictions Table
----------------------------

In [0]:
pred_df = spark.read.table(f'{catalog}.{gold_schema}.future_game_predictions')
display(pred_df)

#Group stage table
--------------------

In [0]:
# World Cup 2026 Groups
groups_dict = {
    # Group A
    'Mexico': 'A',
    'South Africa': 'A',
    'South Korea': 'A',

    
    # Group B
    'Canada': 'B',
    'Qatar': 'B',
    'Switzerland': 'B',
    
    # Group C
    'Brazil': 'C',
    'Morocco': 'C',
    'Haiti': 'C',
    'Scotland': 'C',
    
    # Group D
    'UAE': 'D',
    'Paraguay': 'D',
    'Australia': 'D',
    'United States' : 'D',

    
    # Group E
    'Germany': 'E',
    'Curaçao': 'E',
    "Côte d'Ivoire": 'E',
    'Ecuador': 'E',
    
    # Group F
    'Netherlands': 'F',
    'Japan': 'F',
    'UEFA Playoff B': 'F',
    'Tunisia': 'F',
    
    # Group G
    'Belgium': 'G',
    'Egypt': 'G',
    'Iran': 'G',
    'New Zealand': 'G',
    
    # Group H
    'Spain': 'H',
    'Cape Verde': 'H',
    'Saudi Arabia': 'H',
    'Uruguay': 'H',
    
    # Group I
    'France': 'I',
    'Senegal': 'I',
    'Norway': 'I',
    
    # Group J
    'Argentina': 'J',
    'Algeria': 'J',
    'Austria': 'J',
    'Jordan': 'J',
    
    # Group K
    'Portugal': 'K',
    'Uzbekistan': 'K',
    'Colombia': 'K',
    
    # Group L
    'England': 'L',
    'Croatia': 'L',
    'Ghana': 'L',
    'Panama': 'L'
}

# Join to get group stage
--------------------------

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf, when, col

# Create UDF to map team to group
def get_group(team):
    return groups_dict.get(team, None)

group_udf = udf(get_group, StringType())

# Add home_group and away_group columns to pred_df
pred_df = pred_df.withColumn('home_group', group_udf(col('home_team'))) \
                 .withColumn('away_group', group_udf(col('away_team')))

# Add group column: if home_group is null, take away_group, else take home_group
pred_df = pred_df.withColumn('group', when(col('home_group').isNull(), col('away_group')).otherwise(col('home_group'))).drop('home_group', 'away_group')

display(pred_df)

# Create the simualtion
----------------------

In [0]:
# Update predicted_outcome to 'Draw' for both conditions
pred_df = pred_df.withColumn(
    'predicted_outcome',
    when(
        ((col('predicted_outcome') == 'Home Win') & ((col('prob_home_win') - col('prob_away_win')) < 0.20) & (col('prob_draw') > 0.25)) |
        ((col('predicted_outcome') == 'Away Win') & ((col('prob_away_win') - col('prob_home_win')) < 0.20) & (col('prob_draw') > 0.25)),
        'Draw'
    ).otherwise(col('predicted_outcome'))
)

display(pred_df)

In [0]:
### Add points gained for each teams (home,away)
simulate_results_df = pred_df.withColumn('home_points', when(col('predicted_outcome') == 'Home Win', 3).when(col('predicted_outcome') == 'Draw', 1).otherwise(0))
simulate_results_df = simulate_results_df.withColumn('away_points', when(col('predicted_outcome') == 'Away Win', 3).when(col('predicted_outcome') == 'Draw', 1).otherwise(0))

In [0]:
simulate_results_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog}.{gold_schema}.final_pred_table')

In [0]:
### Make team unified and points then keeo group for grouping later on results.
home_df = simulate_results_df.select('id','home_team','home_points','group').withColumnRenamed('home_team','team').withColumnRenamed('home_points','points')
away_df = simulate_results_df.select('id','away_team','away_points','group').withColumnRenamed('away_team','team').withColumnRenamed('away_points','points')
# Combine home_df and away_df
combined_df = home_df.unionByName(away_df)

# Create grouped results
-------------------------

In [0]:
### Group by the team and group and aggregate sum of points

grouped_results = combined_df.groupBy('group','team').agg({'points': 'sum'}).withColumnRenamed('sum(points)', 'points').orderBy('group','points',ascending=[True,False])
display(grouped_results)

grouped_results.write.format('delta').mode('overwrite').saveAsTable(f'{catalog}.{gold_schema}.group_stage_results')